In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

In [ ]:
df = pd.read_excel("Data/datos_limpios.xlsx")
df

In [ ]:
print("Número de observaciones por clase")
print(df['Impago'].value_counts())
print("")

print("Porcentaje de observaciones por clase")
print(100 * df['Impago'].value_counts(normalize=True))


El 88% de los correos no son spam y el 11% sí lo son. Un modelo de clasificación que sea útil debe de ser capaz de predecir correctamente un porcentaje de observaciones por encima del porcentaje de la clase mayoritaria. En este caso, el umbral de referencia que se tiene que superar es del 88%.

In [ ]:

X = df[['Edad', 'Monto_Inicial', 'Scoring_Crediticio',
       'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes','Ratio_Deuda_Ingresos', 'Estudios', 'Posesion_Hipoteca',
       'Personas_Cargo', 'Fiador', 'Prima']] 
y = df['Impago']

X_train, X_test, y_train, y_test = train_test_split(
                                        X, y.values,
                                        train_size   = 0.7,
                                        random_state = 42,
                                        shuffle      = True)

In [ ]:
# Escalado de variables

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

In [ ]:
modelo_log_sklearn = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
modelo_log_sklearn.fit(X_train_scaled, y_train)

In [ ]:
# Información del modelo
print("Intercept:", modelo_log_sklearn.intercept_)
print("Coeficiente:", list(zip(X.columns, modelo_log_sklearn.coef_.flatten(), )))
print("Accuracy de entrenamiento:", modelo_log_sklearn.score(X_train, y_train))

In [ ]:
# predict_proba()
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

predicciones = modelo_log_sklearn.predict_proba(X = X_test_scaled)
predicciones = pd.DataFrame(predicciones, columns = modelo_log_sklearn.classes_)
predicciones.head(5)

In [ ]:
# Creación del modelo utilizando matrices como sklearn

X_train_scaled = sm.add_constant(X_train_scaled, prepend=True)
modelo_log_statsm = sm.Logit(endog= y_train, exog = X_train_scaled)
modelo_log_statsm = modelo_log_statsm.fit()
print(modelo_log_statsm.summary())

In [ ]:
intervalos_ci = modelo_log_statsm.conf_int(alpha=0.05).rename(columns={0: "2.5%", 1: "97.5%"})

intervalos_ci[:4]

In [ ]:
# Debemos añadir una columna de constante al conjunto de test 

X_test_scaled = sm.add_constant(X_test_scaled, has_constant='add')

predicciones = modelo_log_statsm.predict(exog = X_test_scaled)
predicciones[:4]

In [ ]:
# # Hallamos la clasificación predicha estableciendo un umbral de probabilidad 0.5

# clasificacion = np.where(predicciones<0.5, 0, 1)
# clasificacion[:4]

In [ ]:
# Quitamos la columna 'const' (añadida por statsmodels) para que sklearn pueda predecir
X_test_sklearn = X_test_scaled.drop(columns=['const'], errors='ignore')

# Obtenemos solo la columna de probabilidades de la clase 1 (Impago)
probabilidades_impago = modelo_log_sklearn.predict_proba(X_test_sklearn)[:, 1]

# Definimos el umbral de decisión (puedes ir bajándolo, ej: 0.20, 0.15...)
umbral_decision = 0.50

# Hallamos la clasificación predicha estableciendo el umbral
clasificacion = np.where(probabilidades_impago < umbral_decision, 0, 1)

print(clasificacion[:4])

### Métricas de rendimiento del modelo

- Accuracy

- precision

- sensibilidad (recall)

- f1-score

- Matriz de confusión

- Curva precisión/sensibilidad

- curva ROC-AUC

In [ ]:
# Accuracy / Aciertos: 
""" 
Mide la proporción de predicciones correctas sobre el total de observaciones evaluadas
"""

from sklearn.metrics import accuracy_score

acierto = accuracy_score(y_test, clasificacion)
error = 1 - acierto
print("Acierto:", round(acierto*100, 2), "%")
print("Error:", round(error*100, 2), "%")

In [ ]:
# Precisión, Recall, f1-score
""" 
Precision: Cada vez que el modelo predijo una clase, acerto ese porcentaje
Recall: De todos los datos que eran de esa clase, detectó ese porcentaje
F1-score: El valor será alto si precisión y sensibilidad son similares, útil para comparar clasificadores.
"""



from sklearn.metrics import classification_report

print(classification_report(y_test, clasificacion))


In [ ]:
# Matriz de confusión

from sklearn.metrics import confusion_matrix

c_matrix = confusion_matrix(y_test, clasificacion)
print(c_matrix)

sns.heatmap(c_matrix, annot=True)

In [ ]:
## Curva precisión/recall

""" 
Cómo varía la precisión y sensibilidad respecto a diferentes umbrales de probabilidad
"""
from sklearn.metrics import precision_recall_curve

# Primero creamos una tabla para ver como varía la precisión y recall en función del valor umbral

prec, rec, thresholds = precision_recall_curve(y_test, predicciones) # necesitamos la probabilidad de la clase 1
pd.DataFrame({"prec":prec[:-1], "rec":rec[:-1], "threshold":thresholds})[80:100]


In [ ]:
# Creamos la curva

plt.plot(rec, prec)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("PR Curve")

In [ ]:
# Curva ROC: 

""" 
- tpr (True Positive Rate / Sensibilidad / Recall): Porcentaje de casos positivos (1) reales que el modelo logra detectar
- fpr (False Positive Rate / 1 - Especificidad): Porcentaje de casos negativos (0) reales que el modelo clasifica erróneamente como positivos.
- threshold: El umbral de probabilidad utilizado para obtener esa combinación específica de tpr y fpr.

"""
from sklearn.metrics import roc_curve

fpr, tpr, threshold = roc_curve(y_test, predicciones)
pd.DataFrame({"tpr":tpr, "fpr":fpr, "threshold":threshold})


In [ ]:
import matplotlib.pyplot as plt

plt.plot(fpr, tpr)

plt.xlabel("Ratio de falsos positivos")
plt.ylabel("Ratio de verdaderos positivos")
plt.title("Curva ROC")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generar y visualizar la nueva matriz de confusión
c_matrix = confusion_matrix(y_test, clasificacion)

plt.figure(figsize=(8, 6))
sns.heatmap(c_matrix, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusión - Riesgo de Impago")
plt.ylabel("Realidad")
plt.xlabel("Predicción")
plt.show()